# mosdif Web Service on Google Colab

Notebook ini disiapkan untuk menjalankan **mosdif** sebagai service web (Gradio) menggunakan GPU A100 di Google Colab Pro, dan di-tunnel ke AWS VM Anda.

> **PENTING:** Pastikan Anda memilih Hardware accelerator **GPU A100** dan **High-RAM** (Runtime > Change runtime type).


### 1. Cek GPU dan Alokasi VRAM
Pastikan GPU yang terdeteksi adalah A100 atau L4 dengan VRAM yang mencukupi.

In [1]:
!nvidia-smi
import torch
print('CUDA Available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('Device:', torch.cuda.get_device_name(0))
    total_mem = torch.cuda.get_device_properties(0).total_memory / (1024**3)
    print(f'VRAM: {total_mem:.2f} GB')

Thu Oct  8 10:45:24 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA L4                      Off |   00000000:00:03.0 Off |                    0 |
| N/A   39C    P8             12W /   72W |       0MiB /  23034MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

### 2. Setup ComfyUI & mosdif Workspace
ComfyUI diperlukan sebagai backend inferensi untuk model MiniMax H3.

In [18]:
!git clone https://github.com/azifypage/mosdif.git

In [4]:
# Clone ComfyUI backend
!git clone https://github.com/comfyanonymous/ComfyUI /content/ComfyUI
!pip install -q -r /content/ComfyUI/requirements.txt

# Pindah ke folder mosdif
%cd /content/mosdif

# Hubungkan custom nodes bawaan mosdif ke ComfyUI
!mkdir -p /content/ComfyUI/custom_nodes
!cp -r /content/mosdif/comfy_nodes/* /content/ComfyUI/custom_nodes/ 2>/dev/null || true
!cp -r /content/MosaicDiff/comfy_nodes/* /content/ComfyUI/custom_nodes/ 2>/dev/null || true


fatal: destination path '/content/ComfyUI' already exists and is not an empty directory.
/content/MosaicDiff


### 3. Install Dependensi mosdif & Gradio

In [5]:
# Install dependensi mosdif, Gradio, dan SageAttention untuk percepatan GPU MiniMax H3
!pip install -q gradio av opencv-python mmengine huggingface_hub onnxruntime-gpu sageattention
!pip install -q --no-build-isolation nvidia-vfx --index-url https://pypi.nvidia.com 2>/dev/null || true


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.8/46.8 kB 4.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 452.7/452.7 kB 40.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 246.7/246.7 MB 8.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 256.2/256.2 kB 27.8 MB/s eta 0:00:00


### 4. Unduh Model & Weights (DI-KOMENTARI / DISABLE SEMENTARA)

**Catatan:** Semua perintah pengunduhan model di bawah ini sengaja diberi tanda komentar (`#`).
Ketika Anda sudah siap mengunduh di Colab nanti, cukup hapus tanda `#` pada baris-baris yang diperlukan.
Total ukuran seluruh model adalah ~37 GB.

In [6]:
# ==========================================================================
# 4. SINKRONISASI / UNDUH MODEL KE mosdif/models
# ==========================================================================

!mkdir -p /content/mosdif/models
%cd /content/mosdif

# 4A. Jika model sudah terunduh di /content/MosaicDiff/models, langsung sambungkan:
!cp -n /content/MosaicDiff/models/* /content/mosdif/models/ 2>/dev/null || true

# 4B. Download detector (rfdetr.onnx) dan custom LoRA (lora.safetensors) jika belum ada:
!if [ ! -f "/content/mosdif/models/rfdetr.onnx" ] || [ ! -f "/content/mosdif/models/lora.safetensors" ]; then \
    wget -q -O MosaicDiff-windows.7z "https://github.com/dotaku22/MosaicDiff/releases/download/v1.0.1/MosaicDiff-windows.7z" && \
    7z e MosaicDiff-windows.7z models/rfdetr.onnx models/lora.safetensors -omodels/ -y && \
    rm MosaicDiff-windows.7z; \
fi

# 4C. Download model publik HuggingFace (BasicVSR, Eros Max, CLIP, VAE):
import threading
from mosaicdiff.fetch import ensure_weights
from mosaicdiff.settings import Settings

settings = Settings.load()
settings.paths["comfy_root"] = "/content/ComfyUI"
settings.paths["comfy_python"] = "/usr/bin/python3"
settings.save()

print("Memeriksa dan mengunduh model...")
ensure_weights(settings, print, threading.Event())
print("Semua model berhasil terdeteksi dan siap!")


/content/MosaicDiff

7-Zip 23.01 (x64) : Copyright (c) 1999-2023 Igor Pavlov : 2023-06-20
 64-bit locale=en_US.UTF-8 Threads:12 OPEN_MAX:1048576

Scanning the drive for archives:
  0M Scan         1 file, 2102307535 bytes (2005 MiB)

Extracting archive: MosaicDiff-windows.7z
--
Path = MosaicDiff-windows.7z
Type = 7z
Physical Size = 2102307535
Headers Size = 81936
Method = LZMA2:27 BCJ
Solid = +
Blocks = 2

  0%      1%      3%      6%      9%     11%     14%     16%     19%     21%     24%     26%     28%     29%     31%     32%     34%     35%     37%     38%     40%     41%     43%     44%     46%     48%     49%     51%     52%     54%     55%     57%     58%     60%     61%     67%    

BasicVSR++ checkpoint: 74.8 MB to download
BasicVSR++ checkpoint: received 12.1 MB / 74.8 MB (16.2%) at 9.1 MB/s, about 6s left
BasicVSR++ checkpoint: saved 31.2 MB / 74.8 MB (41.8%) at 17.1 MB/s, about 2s left
BasicVSR++ checkpoint: saved 74.8 MB / 74.8 MB (100.0%) at 712.2 MB/s
BasicVSR++ checkpoint: received 67.4 MB / 67.4 MB (100.0%) at 98.2 MB/s
BasicVSR++ checkpoint: saved 74.8 MB / 74.8 MB (100.0%) at starting
Saved basicvsr.pth
Eros Max: 19.5 GB to download
Eros Max: received 55.6 KB / 19.5 GB (0.0%) at 36.5 KB/s, about 155 h 38 min left
Eros Max: saved 1.3 MB / 19.5 GB (0.0%) at 819.6 KB/s, about 6 h 56 min left
Eros Max: received 981.0 MB / 19.5 GB (4.9%) at 325.0 MB/s, about 58s left
Eros Max: saved 736.8 MB / 19.5 GB (3.7%) at 243.6 MB/s, about 1 min left
Eros Max: received 1.6 GB / 19.5 GB (8.1%) at 214.1 MB/s, about 1 min left
Eros Max: saved 1.5 GB / 19.5 GB (7.7%) at 222.0 MB/s, about 1 min left
Eros Max: received 1.8 GB / 19.5 GB (9.5%) at 88.0 MB/s, about 3 min left
E

### 5. Setup Reverse SSH Tunnel ke AWS VM
Tunnel ini akan menghubungkan port `7860` di Colab ke IP VM AWS Anda.

**Konfigurasi di AWS VM:**
1. Pastikan port 80 (atau port yang Anda gunakan) di-allow di AWS Security Group (Inbound Rule).
2. Di file `/etc/ssh/sshd_config` AWS VM, pastikan ada baris: `GatewayPorts yes` lalu jalankan `sudo systemctl restart ssh`.
3. Upload private key (`.pem`) Anda ke Google Colab (misalnya ke `/content/aws_key.pem`).

In [24]:
AWS_PUBLIC_IP="3.26.217.185"
SSH_USER="ec2-user"
PEM_PATH="/content/aws_key.pem"

!chmod 600 {PEM_PATH}

!nohup ssh \
    -o StrictHostKeyChecking=no \
    -o ServerAliveInterval=30 \
    -o ServerAliveCountMax=3 \
    -N \
    -R 127.0.0.1:7860:localhost:7860 \
    {SSH_USER}@{AWS_PUBLIC_IP} \
    -i {PEM_PATH} \
    > /content/ssh-tunnel.log 2>&1 &

### 6. Jalankan mosdif Web Service (Gradio)
Jalankan web service pada port 7860. Jika tunnel aktif, Anda bisa langsung mengaksesnya lewat browser menggunakan IP AWS.

In [21]:
%cd /content/mosdif

# Jalankan WebUI di background dengan nohup
!nohup python webui.py --host 0.0.0.0 --port 7860 > /content/mosdif.log 2>&1 &

# Cek proses
!sleep 3
!ps aux | grep webui.py | grep -v grep

# Cek apakah port 7860 sudah listening
!curl -I http://localhost:7860

HTTP/1.1 200 OK
date: Thu, 08 Oct 2026 11:27:07 GMT
server: uvicorn
content-length: 55641
content-type: text/html; charset=utf-8

